# 2. (3 points)

Given the following information as of December 31, 2008:

In [1]:
import chainladder as cl
import numpy as np
import pandas as pd

# values in thousands ($000)
tri = cl.Triangle(
    data={
        "origin": [2016, 2016, 2016, 2017, 2017, 2018],
        "development": [2016, 2017, 2018, 2017, 2018, 2018],
        "Paid Loss": [1800, 700, 500, 2000, 800, 2200],
    },
    origin="origin",
    development="development",
    columns = ["Paid Loss"],
    cumulative=False
)

earned_premium = pd.Series(data=[5000, 6000, 8000 ], index=[2016, 2017, 2018])

print(tri)
print(earned_premium)

          12     24     36
2016  1800.0  700.0  500.0
2017  2000.0  800.0    NaN
2018  2200.0    NaN    NaN
2016    5000
2017    6000
2018    8000
dtype: int64


- Assume there is no further development after 36 months.
- $Var[U_i] = Var[U^{BC}_{i}]$

* a. (1.5 points) Calculate the accident year 2008 Benktander reserve estimate $(R^{GB})$
* b. (0.5 point) Calculate the accident year 2008 optimal credible reserve estimate $(R_c)$
* c. (0.5 point) Identify which of $R_c$ or $R^{GB}$ is the preferrable reserve estimate from a statistical point of view and briefly describe a supporting reason.
* d. (0.5 point) Describe the effect on the Benktander credibility for accident year 2018 if the incremental paid loss from 12 to 24 months for accident year 2017 was greater than the value in the table above.

## Solution (Hürlimann, *Credible Loss Ratio Claims Reserves*)

**Input notes.** The question says "as of December 31, 2008" and "accident year 2008", but the data are labeled 2016–2018. I treat the label as a relabeling: the "accident year 2008" question is about the **latest accident year (2018, 12 months of development)**. No expected loss ratio is given, and none is needed: Hürlimann's collective (burning-cost) estimate is $U^{BC}_i = V_i\sum_k \hat m_k$, where $\hat m_k$ is the premium-weighted incremental loss-ratio pattern from the triangle itself.

Method (all from Hürlimann):
- $\hat m_k = \sum_i S_{ik} / \sum_i V_i$ over the origins observed at age $k$; $p_i=\sum_{k\le n-i+1}\hat m_k/\sum_k\hat m_k$, $q_i=1-p_i$ (eq. 2.3-2.4).
- Individual reserve $R^{ind}=\frac{q}{p}C$; collective reserve $R^{coll}=q\,U^{BC}$ (2.5-2.7).
- Benktander: $R^{GB}=p\,R^{ind}+q\,R^{coll}$ (3.2).
- Optimal credible reserve: $R_c=Z R^{ind}+(1-Z)R^{coll}$, $Z=p/(p+t)$. Given $Var[U_i]=Var[U^{BC}_i]$, i.e. $f_i=1$, the minimum-variance choice is $t^*_i=\sqrt{p_i}$ (eq. 6.1, Theorem 6.1), so $Z^*=\sqrt{p}/(1+\sqrt{p})$.

In [2]:
cum = tri.incr_to_cum().to_frame()   # cumulative paid by origin x age
inc = tri.to_frame()
V = earned_premium.values.astype(float)

# premium-weighted incremental loss-ratio pattern m_k (sum of incrementals / sum of premium of origins observed at age k)
m = np.array([inc.iloc[:, k].sum() / V[inc.iloc[:, k].notna().values].sum() for k in range(3)])
p = m.cumsum() / m.sum()          # p for an origin at age k+1
print("m_k:", m.round(5), " p:", p.round(5))

p18, q18 = p[0], 1 - p[0]         # AY 2018 is at 12 months
C18, V18 = 2200.0, 8000.0
U_bc = V18 * m.sum()
R_coll = q18 * U_bc
R_ind = q18 / p18 * C18
R_gb = p18 * R_ind + q18 * R_coll
Z = np.sqrt(p18) / (1 + np.sqrt(p18))
R_c = Z * R_ind + (1 - Z) * R_coll
print(f"U_BC={U_bc:,.1f}  R_coll={R_coll:,.1f}  R_ind={R_ind:,.1f}")
print(f"a. R_GB = {R_gb:,.1f}")
print(f"b. Z* = {Z:.4f}   R_c = {R_c:,.1f}")
assert np.isclose(R_gb, 1751.2, atol=0.05) and np.isclose(R_c, 1785.7, atol=0.05)

m_k: [0.31579 0.13636 0.1    ]  p: [0.57192 0.81889 1.     ]
U_BC=4,417.2  R_coll=1,890.9  R_ind=1,646.7
a. R_GB = 1,751.2
b. Z* = 0.4306   R_c = 1,785.7


**a.** $p_{2018}=0.3158/0.5521=0.5719$, $q=0.4281$. $R^{ind}=0.4281/0.5719\times2{,}200=1{,}646.7$; $R^{coll}=0.4281\times8{,}000\times0.5521=1{,}890.9$.
$R^{GB}=0.5719(1{,}646.7)+0.4281(1{,}890.9)=$ **≈ 1,751 ($000)**.

**b.** $t^*=\sqrt{0.5719}=0.7563$, $Z^*=0.5719/(0.5719+0.7563)=0.4306$.
$R_c=0.4306(1{,}646.7)+0.5694(1{,}890.9)=$ **≈ 1,786 ($000)**.

**c.** $R_c$ is preferable statistically. Its credibility weight is chosen to minimize the mean squared error of the reserve (and, with $f=1$, also its variance), whereas the Benktander weight $Z=p$ is an ad hoc choice that is not optimal. (Here $Z^{GB}=0.572 > Z^*=0.431$, so Benktander puts more weight on the more volatile individual reserve; Hürlimann notes $Z^*\le 1/2$ always.)

**d.** Benktander credibility for AY 2018 is $Z^{GB}_{2018}=p_{2018}=\sum_{k=1}^{1}\hat m_k/\sum_k\hat m_k$. A larger AY 2017 incremental paid loss at 12–24 months raises $\hat m_2$, which raises the total $\sum\hat m_k$ and leaves $\hat m_1$ unchanged, so $p_{2018}$ **decreases**, and so does the Benktander credibility (more weight shifts to the collective reserve). (The optimal $Z^*=\sqrt p/(1+\sqrt p)$ falls too.)

In [3]:
# Check d numerically: bump AY2017 12-24 incremental from 800 to 1000
inc2 = inc.copy(); inc2.iloc[1, 1] = 1000
m2 = np.array([inc2.iloc[:, k].sum() / V[inc2.iloc[:, k].notna().values].sum() for k in range(3)])
print("p_2018 before/after:", round(p18, 4), round(m2[0] / m2.sum(), 4))
assert m2[0] / m2.sum() < p18

p_2018 before/after: 0.5719 0.5537


## chainladder-python gaps

- `cl.CapeCod` derives the ELR from the data, but `cl.Benktander` needs the a priori as an input and the two do not connect. Hürlimann's collective estimate is a Cape Cod estimate (premium-weighted burning cost $\sum\hat m_k$ = latest paid / $\sum V_ip_i$) with the pattern estimated from loss ratios. The pattern in chainladder comes from age-to-age LDFs (volume average gives $p_{2018}=0.5975$ vs Hürlimann's loss-ratio pattern $0.5719$), so `Benktander(apriori=sum(m), n_iters=1)` gives an AY2018 IBNR of 1,778 rather than 1,751. (The difference comes from the pattern, not a bug.)
- No **loss-ratio (premium-weighted incremental) payout pattern** estimator, and no individual/collective loss ratio reserving methods (Mack 1997 / Hürlimann §2).
- No **credibility-weighted reserve** with optimal weights (Hürlimann $Z^*=p/(p+t^*)$, $t^*=\sqrt p$; Neuhaus; or the general eq. 5.5) and no **MSE of prediction** (Theorems 4.3, 6.1), nor the pragmatic variance estimators (5.1-5.2).
- Suggested feature: `cl.HurlimannCLR(method='optimal'|'benktander'|'neuhaus', f=1)` fit on a triangle plus `sample_weight=premium`, with a `pattern='loss_ratio'` option that estimates $\hat m_k$ from premium, and attributes `credibility_`, `ibnr_`, `mse_`. Alternatively `Benktander(apriori='burning_cost')`.

## Sketch: what a `HurlimannCLR` estimator could look like

Below is a **prototype written in the notebook** (not part of chainladder) with the interface I'd propose: sklearn-style `fit(X, sample_weight=premium)`, then attributes. It first shows how the existing `cl.CapeCod` compares.

In [4]:
# Premium as a chainladder Triangle, as Benktander/CapeCod expect for sample_weight
sw = cl.Triangle(
    pd.DataFrame({"origin": [2016, 2017, 2018], "development": [2018] * 3, "EP": earned_premium.values}),
    origin="origin", development="development", columns=["EP"], cumulative=True,
)
cum_tri = tri.incr_to_cum()

# Existing feature: Cape Cod derives the ELR, but pattern comes from LDFs
dev = cl.Development(average="volume").fit_transform(cum_tri)
cc = cl.CapeCod(trend=0, decay=1).fit(dev, sample_weight=sw)
print("CapeCod ELR (LDF pattern):", cc.apriori_.to_frame().values.ravel()[:3].round(4))
print("CapeCod IBNR:", cc.ibnr_.to_frame().values.ravel().round(1))
print("Hurlimann collective ELR :", round(m.sum(), 4), " AY2018 R_coll:", round(R_coll, 1))

CapeCod ELR (LDF pattern): [0.5413 0.5413 0.5413]
CapeCod IBNR: [   nan  541.3 1743. ]
Hurlimann collective ELR : 0.5522  AY2018 R_coll: 1890.9


In [5]:
from sklearn.base import BaseEstimator


class HurlimannCLR(BaseEstimator):
    """Prototype of Hurlimann's credible loss ratio reserving.

    method : 'optimal' (Z = p/(p+sqrt(p)), f=1), 'benktander' (Z = p),
             'collective' (Z = 0), 'individual' (Z = 1)
    """

    def __init__(self, method="optimal"):
        self.method = method

    def fit(self, X, y=None, sample_weight=None):
        cum = X.values[0, 0]                          # origin x age, cumulative, NaN below diagonal
        inc = np.diff(cum, axis=1, prepend=0)         # NaN propagates
        V = sample_weight.latest_diagonal.values[0, 0, :, 0]
        seen = ~np.isnan(inc)
        self.m_ = np.nansum(inc, axis=0) / (seen * V[:, None]).sum(axis=0)     # loss ratio pattern
        n_obs = seen.sum(axis=1)
        self.p_ = self.m_.cumsum()[n_obs - 1] / self.m_.sum()                  # % developed by origin
        self.elr_ = self.m_.sum()                                              # burning-cost ELR
        C = np.array([cum[i, n_obs[i] - 1] for i in range(len(V))])
        q = 1 - self.p_
        self.r_individual_ = q / self.p_ * C
        self.r_collective_ = q * V * self.elr_
        self.credibility_ = {
            "optimal": np.sqrt(self.p_) / (1 + np.sqrt(self.p_)),
            "benktander": self.p_,
            "collective": np.zeros_like(self.p_),
            "individual": np.ones_like(self.p_),
        }[self.method]
        self.ibnr_ = self.credibility_ * self.r_individual_ + (1 - self.credibility_) * self.r_collective_
        self.ultimate_ = C + self.ibnr_
        return self

    def summary(self, index):
        return pd.DataFrame({"p": self.p_, "Z": self.credibility_, "R_ind": self.r_individual_,
                             "R_coll": self.r_collective_, "IBNR": self.ibnr_, "Ultimate": self.ultimate_}, index=index)

In [6]:
for method in ("benktander", "optimal"):
    est = HurlimannCLR(method=method).fit(cum_tri, sample_weight=sw)
    print(method); print(est.summary(earned_premium.index).round(3), "\n")

assert np.isclose(HurlimannCLR("benktander").fit(cum_tri, sample_weight=sw).ibnr_[-1], R_gb)
assert np.isclose(HurlimannCLR("optimal").fit(cum_tri, sample_weight=sw).ibnr_[-1], R_c)

benktander
          p      Z     R_ind    R_coll      IBNR  Ultimate
2016  1.000  1.000     0.000     0.000     0.000  3000.000
2017  0.819  0.819   619.259   600.000   615.771  3415.771
2018  0.572  0.572  1646.667  1890.909  1751.221  3951.221 

optimal
          p      Z     R_ind    R_coll      IBNR  Ultimate
2016  1.000  0.500     0.000     0.000     0.000  3000.000
2017  0.819  0.475   619.259   600.000   609.149  3409.149
2018  0.572  0.431  1646.667  1890.909  1785.737  3985.737 



In practice this would sit beside the existing methods, e.g.

```python
cl.HurlimannCLR(method="optimal").fit(triangle, sample_weight=premium).ibnr_
cl.Pipeline([("dev", cl.LossRatioPattern()), ("cred", cl.HurlimannCLR())])   # pattern as a separate step
```

Natural extensions not prototyped here: `mse_` (Hürlimann Theorem 4.3; needs the pragmatic estimates in eqs. 5.1-5.4), `method="neuhaus"`, the `f` and `beta` parameters of eq. 5.5, and exposing `elr_` so it can seed `cl.Benktander`/`cl.BornhuetterFerguson` as `apriori`.